# ✈️ Airline Fare Intelligence — Adapted Colab Project
**Data:** Clean_Dataset.csv. **Goal:** defensible fare regression, 16 transparent charts, tests, reports and a Streamlit app.

**Important:** This dataset cannot establish dated 7-day fare changes, festival impacts or backtested savings. The project deliberately does not fabricate them.

## 1. Setup — upload project ZIP if necessary

In [ ]:
!pip -q install lightgbm streamlit pytest joblib shap
from pathlib import Path
import os, sys, zipfile
ROOT=Path('/content/airline_fare_intelligence')
if not (ROOT/'src/train.py').exists():
    from google.colab import files
    print('Upload airline_fare_intelligence_project.zip (provided with this notebook)')
    uploaded=files.upload()
    z=next((n for n in uploaded if n.endswith('.zip')),None)
    if not z: raise FileNotFoundError('Upload the project ZIP file to continue')
    with zipfile.ZipFile(z) as archive: archive.extractall('/content')
os.chdir(ROOT);sys.path.insert(0,str(ROOT))
print('Project root:',ROOT)

## 2. Dataset audit

In [ ]:
import pandas as pd
from src.features import clean_and_engineer
p=ROOT/'data/raw/Clean_Dataset.csv'
if not p.exists():
    from google.colab import files
    uploaded=files.upload()
    p.write_bytes(next(iter(uploaded.values())))
raw=pd.read_csv(p)
df=clean_and_engineer(raw)
print('Rows:',len(df),'Columns:',df.shape[1],'Routes:',df.route.nunique(),'Flights:',df.flight.nunique())
display(df.head(),df.isna().sum().to_frame('missing'))

## 3. Train and evaluate (flight-number-disjoint holdout)

In [ ]:
from src.train import train
df,test_predictions,metrics,best_model=train()
display(metrics)

## 4. Produce 16 honest visualizations

In [ ]:
from src.visualize import make_charts
make_charts(df,test_predictions,metrics,best_model)
from IPython.display import display,Image
for p in sorted((ROOT/'outputs/figures').glob('chart*.png')):
    print(p.name)
    display(Image(filename=str(p),width=650))

## 5. Automated tests

In [ ]:
!PYTHONPATH=/content/airline_fare_intelligence python -m pytest -q tests

## 6. Example inference

In [ ]:
from src.predict import predict_price
row=df.iloc[0]
features=['airline','source_city','destination_city','departure_time','arrival_time','stops','class','duration','days_left']
print('Predicted INR:',round(predict_price(row[features].to_dict()),2))

## 7. Report and export

In [ ]:
import json, shutil
print((ROOT/'outputs/reports/summary.json').read_text())
archive=shutil.make_archive('/content/airline_fare_intelligence_completed','zip',root_dir='/content',base_dir='airline_fare_intelligence')
print('Completed ZIP:',archive)
from google.colab import files
files.download(archive)

## 8. Streamlit app
Run locally with `streamlit run app/app.py`. In Colab, a public tunnel requires additional setup; the app source is included in the ZIP.

### Project limitations
No chronological holdout, rise classification, festival analysis, genuine buy/wait strategy or regret metric is claimed. These require additional dated fare snapshots.